### Importing the Required Libraries

In [1]:
# import sys
# !{sys.executable} -m pip install --upgrade --force-reinstall langchain langchain-community langchain-huggingface
import langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import pipeline
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline

C:\Users\rajve\miniconda3\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Reading The Document 

In [2]:
loader=TextLoader("companyPolicies.txt")
document=loader.load()

In [3]:
document

[Document(metadata={'source': 'companyPolicies.txt'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to 

### Chunking Of The Document

In [4]:
splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=0)
document=splitter.split_documents(document)

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


In [5]:
document

[Document(metadata={'source': 'companyPolicies.txt'}, page_content='1.\tCode of Conduct'),
 Document(metadata={'source': 'companyPolicies.txt'}, page_content="Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decis

### Creating Embeddings And Storing In Chroma DB 

In [6]:
embeddings=HuggingFaceEmbeddings()
docsearch=Chroma.from_documents(document,embeddings,persist_directory='.\chroma_embeddings')

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 4499.65it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
docsearch.embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Loading The LLM 

In [8]:
model=pipeline(model='unsloth/Llama-3.2-3B-Instruct',
               task='text-generation')

Loading weights: 100%|█████████████████████████████████████████████████████████████| 254/254 [00:00<00:00, 2132.46it/s]


### Making The Prompt Template 

In [9]:
template="Only Give the answer with respect to {context} of the {question} do not try to make it at any cost if answer not given in {context} just say don't know the answer"

In [10]:
from langchain_core.prompts import PromptTemplate
proTemp=PromptTemplate(template=template,input_variables=["context", "question"])

### Building a QA Chain 

In [14]:
rag_chain = (
    {"context": docsearch.as_retriever(), "question": RunnablePassthrough()}
    | proTemp 
    | (lambda x: x.to_string())
    | model 
)

### Testing The Agent 

In [ ]:
rag_chain.invoke("Can I harass a person on Company's Mail ?")